In [4]:
# Import all the libraies
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings

print("--- Rab nee bana diya ML model  ---")

--- Rab nee bana diya ML model  ---


In [6]:
# Read the data. 
try:
    df = pd.read_csv('/Users/tejassaxena/Documents/Database/diabetic_data.csv')
except FileNotFoundError:
    print("Error: 'diabetic_data.csv' not found. Please upload the file.")

# Replace missing values
df = df.replace('?', np.nan)

#Drop colums with too many missing values
cols_to_drop = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
df = df.drop(columns=cols_to_drop, errors='ignore')

In [7]:
fill_values = {}
for col in df.columns:
    if df[col].dtype == 'object':
        fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
        fill_values[col] = fill_val
    else:
        fill_val = df[col].median()
        df[col] = df[col].fillna(fill_val)
        fill_values[col] = fill_val

In [9]:
# Declare target variable
df['target'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# y is the answer key 
X = df.drop(columns=['readmitted', 'target'])
y = df['target']

# Converts catagorial data into numbers so it can be processed
encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le
    

In [11]:
# Feature selection to get top 15
rf_selector = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_selector.fit(X, y)
feature_importances = pd.Series(rf_selector.feature_importances_, index=X.columns)
feature_columns = feature_importances.nlargest(15).index.tolist()
print("Top 15 Features:", feature_columns)

Top 15 Features: ['number_inpatient', 'diag_1', 'diag_2', 'num_lab_procedures', 'diag_3', 'num_medications', 'discharge_disposition_id', 'time_in_hospital', 'age', 'number_diagnoses', 'number_emergency', 'num_procedures', 'admission_type_id', 'insulin', 'number_outpatient']


In [13]:
# Reduce X to only the top 15 features
X = X[feature_columns]

rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

# Training the model 
train_probs = rf_model.predict_proba(X)[:, 1]
raw_threshold = np.percentile(train_probs, 85) 

#and cutoff for risky and non risky patients
raw_min = train_probs.min()
raw_max = train_probs.max()


print("Model Trained Successfully! ")

Model Trained Successfully! 


In [14]:
# Calculate a score and scale it
def calculate_risk_score(raw_prob):
    
    if raw_prob >= raw_threshold:

        if raw_max == raw_threshold: 
            return 99
        scale = (raw_prob - raw_threshold) / (raw_max - raw_threshold)
        score = 70 + (scale * 29)
    else:

        if raw_threshold == raw_min:
            return 1
        scale = (raw_prob - raw_min) / (raw_threshold - raw_min)
        score = 1 + (scale * 59)
        
    return score

In [15]:
 # Takes data and converts it into a format that the model can understand
def predict_patient_readmission(patient_data):

    input_df = pd.DataFrame([patient_data])
    

    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = fill_values.get(col)
            
    for col, le in encoders.items():
        if col in input_df.columns:
            val = str(input_df.iloc[0][col])

            if val in le.classes_:
                input_df[col] = le.transform([val])
            else:
                input_df[col] = le.transform([le.classes_[0]]) 
                
    input_df = input_df[feature_columns]
    raw_prob = rf_model.predict_proba(input_df)[0][1]
    risk_score = calculate_risk_score(raw_prob)
# Finally returns a percentage - (This is the final probability) 
    return risk_score

In [16]:
# Testing on a risky patient [Outputs high risk of coming back as expected.]
new_patient_unhealthy = {
    'race': 'AfricanAmerican', 'gender': 'Female', 'age': '[80-90)',
    'admission_type_id': 1, 'time_in_hospital': 14, 
    'num_lab_procedures': 70, 'num_medications': 30,
    'number_inpatient': 5, 'number_emergency': 2, 
    'number_diagnoses': 9, 'metformin': 'No', 
    'insulin': 'Down', 'change': 'Ch', 'diabetesMed': 'Yes',
    'diag_1': '428' 
}

score_2 = predict_patient_readmission(new_patient_unhealthy)

print("--- PATIENT READMISSION PREDICTIONS ---")

print(f"Patient 2")
print(f"  Risk Score: {score_2:.2f}%")


print("Test report summary.")
print("According to the doctor they ate too many chocolates because they had multiple partners.")

--- PATIENT READMISSION PREDICTIONS ---
Patient 2
  Risk Score: 75.30%
Test report summary.
According to the doctor they ate too many chocolates because they had multiple partners.


In [18]:
# Testing on a healthy patient [Outputs low risk of coming back as expected.]
new_patient_healthy = {
    'race': 'Caucasian', 'gender': 'Male', 'age': '[50-60)',
    'admission_type_id': 1, 'time_in_hospital': 2, 
    'num_lab_procedures': 15, 'num_medications': 5, 
    'number_diagnoses': 3, 'metformin': 'No', 
    'insulin': 'No', 'change': 'No', 'diabetesMed': 'No',
    'diag_1': '250'
}

score_1 = predict_patient_readmission(new_patient_healthy)

print("--- PATIENT READMISSION PREDICTIONS ---")

print(f"Patient 1 ")
print(f"  Risk Score: {score_1:.2f}%")


print("Test report summary.")
print("Staying loyal to one partner was the right thing to do.")

--- PATIENT READMISSION PREDICTIONS ---
Patient 1 
  Risk Score: 12.47%
Test report summary.
Staying loyal to one partner was the right thing to do.


In [ ]:
import pickle

pickle.dump(rf_model, open("diabetes_model.pkl", "wb"))
